<a href="https://colab.research.google.com/github/GarodGB/Soc-project/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving synthetic_dirty_transactions_v4.xlsx to synthetic_dirty_transactions_v4.xlsx


In [3]:
path = "synthetic_dirty_transactions_v4.xlsx"

df=pd.read_excel(path)
df.head()

,ACCOUNT_ID,TXN_ID,TXN_DATE_TIME,SETTLE_DATE,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,PROCESSING_CODE,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS
0,716075454,4700058,2022-03-10 00:51:36,2022-03-13,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,5411,LB,ZAHLE,0,Purchase,PDYR05FW,0MGT4L,Match
1,551324309,4700106,09/07/2022 17:49,10/07/2022,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,5999,GB,LONDON,0,Purchase,D8TG2V7K,RZF979,Match
2,749053287,4700140,2022-09-14 00:09:34,16/09/2022,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,7538,US,ECOM,0,Purchase,00000000,326LGX,Match
3,671284639,4700183,04-22-2022 22:52,23-Apr-22,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,7538,US,E-COMMERCE,0,Purchase,00000000,8TRUC6,Match
4,649925833,4700220,2022-06-24 21:19:59,2022-06-25,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,4111,AE,DUBAYY,0,Purchase,PAXKQGM8,USH2SB,Match


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2296 entries, 0 to 2295
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ACCOUNT_ID        2296 non-null   int64  
 1   TXN_ID            2296 non-null   int64  
 2   TXN_DATE_TIME     2296 non-null   object 
 3   SETTLE_DATE       2293 non-null   object 
 4   TXN_TYPE          2296 non-null   object 
 5   TXN_AMOUNT        2296 non-null   object 
 6   TXN_CCY           2296 non-null   object 
 7   BILLING_AMOUNT    2296 non-null   float64
 8   BILLING_CURRENCY  2296 non-null   object 
 9   FX_RATE           2296 non-null   float64
 10  MERCHANT_NAME     2296 non-null   object 
 11  MCC_CODE          2296 non-null   int64  
 12  MERCHANT_COUNTRY  2296 non-null   object 
 13  MERCHANT_CITY     2229 non-null   object 
 14  PROCESSING_CODE   2296 non-null   int64  
 15  PROCESSING_TYPE   2296 non-null   object 
 16  TERMINAL_ID       2296 non-null   object 


In [5]:
df['date_pattern']=df['TXN_DATE_TIME'].astype(str)
df['date_pattern'] = df['TXN_DATE_TIME'].str.replace(r'\d', '9', regex=True)
df['date_pattern'].value_counts()


,count
date_pattern,
9999-99-99 99:99:99,1214
99/99/9999 99:99,466
99-99-9999 99:99,328
9999999999999,113
99-Jul-99 99:99,28
99-Jun-99 99:99,26
99-Mar-99 99:99,26
99-Aug-99 99:99,25
99-Apr-99 99:99,25


In [6]:
def check_style(pattern, first_pos, second_pos):

    rows = df.loc[df['date_pattern'] == pattern, 'TXN_DATE_TIME'].astype(str)


    first  = rows.str[first_pos[0]:first_pos[1]].astype(int)
    second = rows.str[second_pos[0]:second_pos[1]].astype(int)

    print(pattern,
          '| rows:', len(rows),
          '| first > 12:', (first > 12).sum(),
          '| second > 12:', (second > 12).sum())

check_style('99/99/9999 99:99',    (0, 2), (3, 5))
check_style('99-99-9999 99:99',    (0, 2), (3, 5))
check_style('9999-99-99 99:99:99', (5, 7), (8, 10))

99/99/9999 99:99 | rows: 466 | first > 12: 301 | second > 12: 0
99-99-9999 99:99 | rows: 328 | first > 12: 0 | second > 12: 203
9999-99-99 99:99:99 | rows: 1214 | first > 12: 0 | second > 12: 765


In [7]:
def swap(pattern, dataframe, column_name):
    condition = dataframe['date_pattern'] == pattern


    dataframe[column_name] = dataframe['TXN_DATE_TIME'].astype(str)

    rows = dataframe.loc[condition, 'TXN_DATE_TIME'].astype(str)


    dataframe.loc[condition, column_name] = (rows.str[3:5] + '/' +
                                             rows.str[0:2] + '/' +
                                             rows.str[6:])
    return dataframe

df = swap('99-99-9999 99:99', df, 'TXN_DT_TEXT')



In [8]:
df['less_date'] = df['TXN_DT_TEXT'].str.replace(r'\d', '9', regex=True)
df['less_date'].value_counts()


,count
less_date,
9999-99-99 99:99:99,1214
99/99/9999 99:99,794
9999999999999,113
99-Jul-99 99:99,28
99-Jun-99 99:99,26
99-Mar-99 99:99,26
99-Aug-99 99:99,25
99-Apr-99 99:99,25
99-Sep-99 99:99,24


In [9]:
df['TXN_DT'] = pd.NaT

cond = df['less_date'] == '9999-99-99 99:99:99'

df.loc[cond, 'TXN_DT'] = pd.to_datetime(
    df.loc[cond, 'TXN_DT_TEXT'], format='%Y-%m-%d %H:%M:%S')

cond2 = df['less_date'] == '99/99/9999 99:99'

df.loc[cond2, 'TXN_DT'] = pd.to_datetime(
    df.loc[cond2, 'TXN_DT_TEXT'], format='%d/%m/%Y %H:%M')

df['TXN_DT'].isna().sum()

np.int64(288)

In [10]:
cond3=df['less_date'].str.contains('[a-zA-Z]', regex=True)

df.loc[cond3, 'TXN_DT'] = pd.to_datetime(
    df.loc[cond3, 'TXN_DT_TEXT'], format='%d-%b-%y %H:%M')

In [11]:
df['TXN_DT'].isna().sum()

np.int64(113)

In [12]:
cond4 = df['TXN_DT'].isna()

numbers = pd.to_numeric(df.loc[cond4, 'TXN_DT_TEXT'])

df.loc[cond4, 'TXN_DT'] = pd.to_datetime(numbers, unit='ms')

print(df['TXN_DT'].isna().sum())
print(df['TXN_DT'].min())
print(df['TXN_DT'].max())


0
2022-03-01 00:19:00
2022-09-30 18:52:02


In [13]:
df = df.drop(columns=['TXN_DATE_TIME'])
df = df.drop(columns=['date_pattern'])
df = df.drop(columns=['less_date'])
df = df.drop(columns=['TXN_DT_TEXT'])

In [14]:
df=df.rename(columns={'TXN_DT': 'TXN_DATE_TIME'})

In [15]:
df.head()

,ACCOUNT_ID,TXN_ID,SETTLE_DATE,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,PROCESSING_CODE,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME
0,716075454,4700058,2022-03-13,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,5411,LB,ZAHLE,0,Purchase,PDYR05FW,0MGT4L,Match,2022-03-10 00:51:36
1,551324309,4700106,10/07/2022,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,5999,GB,LONDON,0,Purchase,D8TG2V7K,RZF979,Match,2022-07-09 17:49:00
2,749053287,4700140,16/09/2022,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,7538,US,ECOM,0,Purchase,00000000,326LGX,Match,2022-09-14 00:09:34
3,671284639,4700183,23-Apr-22,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,7538,US,E-COMMERCE,0,Purchase,00000000,8TRUC6,Match,2022-04-22 22:52:00
4,649925833,4700220,2022-06-25,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,4111,AE,DUBAYY,0,Purchase,PAXKQGM8,USH2SB,Match,2022-06-24 21:19:59


In [16]:
df['set_date']=df['SETTLE_DATE'].astype(str)
df['set_date'] = df['set_date'].str.replace(r'\d', '9', regex=True)
df['set_date'].value_counts(dropna=False)

,count
set_date,
9999-99-99,1290
99/99/9999,527
99-99-9999,284
99-Mar-99,32
99-Aug-99,32
99-Jul-99,29
99-Apr-99,27
99-May-99,26
99-Sep-99,23


In [17]:
def check_style_new(pattern, first_pos, second_pos):

    rows = df.loc[df['set_date'] == pattern, 'SETTLE_DATE'].astype(str)


    first  = rows.str[first_pos[0]:first_pos[1]].astype(int)
    second = rows.str[second_pos[0]:second_pos[1]].astype(int)

    print(pattern,
          '| rows:', len(rows),
          '| first > 12:', (first > 12).sum(),
          '| second > 12:', (second > 12).sum())

check_style_new('99/99/9999',    (0, 2), (3, 5))
check_style_new('99-99-9999',    (0, 2), (3, 5))
check_style_new('9999-99-99', (5, 7), (8, 10))

99/99/9999 | rows: 527 | first > 12: 329 | second > 12: 0
99-99-9999 | rows: 284 | first > 12: 0 | second > 12: 168
9999-99-99 | rows: 1290 | first > 12: 0 | second > 12: 799


In [18]:
df['SETTLE_dt'] = pd.NaT

cond = df['set_date'] == '9999-99-99'

df.loc[cond, 'SETTLE_dt'] = pd.to_datetime(
    df.loc[cond, 'SETTLE_DATE'], format='%Y-%m-%d',errors='coerce')

cond2 = df['set_date'] == '99/99/9999'

df.loc[cond2, 'SETTLE_dt'] = pd.to_datetime(
    df.loc[cond2, 'SETTLE_DATE'], format='%d/%m/%Y',errors='coerce')

In [19]:
bad1 = cond & df['SETTLE_dt'].isna()
df.loc[bad1, 'SETTLE_DATE'].value_counts()

,count
SETTLE_DATE,
0000-00-00,9


In [20]:
bad2 = cond2 & df['SETTLE_dt'].isna()
df.loc[bad2, 'SETTLE_DATE'].value_counts()

,count
SETTLE_DATE,


In [21]:

cond3 = df['set_date'] == '99-99-9999'
df.loc[cond3, 'SETTLE_dt'] = pd.to_datetime(
    df.loc[cond3, 'SETTLE_DATE'], format='%m-%d-%Y', errors='coerce')


cond4 = df['set_date'].str.contains('[a-zA-Z]') & (df['set_date'] != 'nan')
df.loc[cond4, 'SETTLE_dt'] = pd.to_datetime(
    df.loc[cond4, 'SETTLE_DATE'], format='%d-%b-%y', errors='coerce')


bad3 = cond3 & df['SETTLE_dt'].isna()
bad4 = cond4 & df['SETTLE_dt'].isna()
print(df.loc[bad3, 'SETTLE_DATE'].value_counts())
print(df.loc[bad4, 'SETTLE_DATE'].value_counts())


print('NaT total:', df['SETTLE_dt'].isna().sum())

Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)
NaT total: 12


In [22]:
txn_day = df['TXN_DATE_TIME'].dt.normalize()
df['SETTLE_DAYS'] = (df['SETTLE_dt'] - txn_day).dt.days

print((df['SETTLE_DAYS'] < 0).sum())
print(df['SETTLE_DAYS'].value_counts().sort_index())

8
SETTLE_DAYS
-19265.0      1
-19221.0      1
-1.0          6
 0.0         37
 1.0        955
 2.0        726
 3.0        382
 4.0        176
Name: count, dtype: int64


In [23]:
problem = df['SETTLE_dt'].isna() | (df['SETTLE_DAYS'] < 0)

print(problem.sum())
print(df.loc[problem, 'TXN_TYPE'].value_counts())
print(df['TXN_TYPE'].value_counts())

20
TXN_TYPE
Settled    19
Refund      1
Name: count, dtype: int64
TXN_TYPE
Settled    2183
Refund      113
Name: count, dtype: int64


In [24]:
good = ~problem
median_days = df.loc[good].groupby('TXN_TYPE')['SETTLE_DAYS'].median()
print(median_days)

TXN_TYPE
Refund     2.0
Settled    2.0
Name: SETTLE_DAYS, dtype: float64


In [25]:
df['SETTLE_IMPUTED'] = problem
fill_days = df.loc[problem, 'TXN_TYPE'].map(median_days)
df.loc[problem, 'SETTLE_dt'] = txn_day[problem] + pd.to_timedelta(fill_days, unit='D')
df['SETTLE_DAYS'] = (df['SETTLE_dt'] - txn_day).dt.days
print( df['SETTLE_dt'].isna().sum())
print( (df['SETTLE_DAYS'] < 0).sum())
print( df['SETTLE_IMPUTED'].sum())

0
0
20


In [26]:
print(df.columns)
df.head()

Index(['ACCOUNT_ID', 'TXN_ID', 'SETTLE_DATE', 'TXN_TYPE', 'TXN_AMOUNT',
       'TXN_CCY', 'BILLING_AMOUNT', 'BILLING_CURRENCY', 'FX_RATE',
       'MERCHANT_NAME', 'MCC_CODE', 'MERCHANT_COUNTRY', 'MERCHANT_CITY',
       'PROCESSING_CODE', 'PROCESSING_TYPE', 'TERMINAL_ID', 'AUTH_CODE',
       'MATCHES_STATUS', 'TXN_DATE_TIME', 'set_date', 'SETTLE_dt',
       'SETTLE_DAYS', 'SETTLE_IMPUTED'],
      dtype='object')


,ACCOUNT_ID,TXN_ID,SETTLE_DATE,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,...,PROCESSING_CODE,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,set_date,SETTLE_dt,SETTLE_DAYS,SETTLE_IMPUTED
0,716075454,4700058,2022-03-13,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,...,0,Purchase,PDYR05FW,0MGT4L,Match,2022-03-10 00:51:36,9999-99-99,2022-03-13,3,False
1,551324309,4700106,10/07/2022,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,...,0,Purchase,D8TG2V7K,RZF979,Match,2022-07-09 17:49:00,99/99/9999,2022-07-10,1,False
2,749053287,4700140,16/09/2022,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,...,0,Purchase,00000000,326LGX,Match,2022-09-14 00:09:34,99/99/9999,2022-09-16,2,False
3,671284639,4700183,23-Apr-22,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,...,0,Purchase,00000000,8TRUC6,Match,2022-04-22 22:52:00,99-Apr-99,2022-04-23,1,False
4,649925833,4700220,2022-06-25,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,...,0,Purchase,PAXKQGM8,USH2SB,Match,2022-06-24 21:19:59,9999-99-99,2022-06-25,1,False


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2296 entries, 0 to 2295
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ACCOUNT_ID        2296 non-null   int64         
 1   TXN_ID            2296 non-null   int64         
 2   SETTLE_DATE       2293 non-null   object        
 3   TXN_TYPE          2296 non-null   object        
 4   TXN_AMOUNT        2296 non-null   object        
 5   TXN_CCY           2296 non-null   object        
 6   BILLING_AMOUNT    2296 non-null   float64       
 7   BILLING_CURRENCY  2296 non-null   object        
 8   FX_RATE           2296 non-null   float64       
 9   MERCHANT_NAME     2296 non-null   object        
 10  MCC_CODE          2296 non-null   int64         
 11  MERCHANT_COUNTRY  2296 non-null   object        
 12  MERCHANT_CITY     2229 non-null   object        
 13  PROCESSING_CODE   2296 non-null   int64         
 14  PROCESSING_TYPE   2296 n

In [28]:
df['TXN_AMOUNT']=df['TXN_AMOUNT'].astype(str).str.replace(r'[^0-9.,-]', '', regex=True)

In [29]:
df['new_ta']=df['TXN_AMOUNT'].astype(str).str.replace(r'\d', '9', regex=True)

In [30]:
df['new_ta'].value_counts()

,count
new_ta,
-999.99,766
-99.99,466
-9999999,246
-9999.99,141
-999999,97
-99999999,95
-9.99,81
-999.9,76
-99,41


In [31]:
import numpy as np

amt = df['TXN_AMOUNT'].astype(str)

conditions = [
    amt.str.contains(','),
    amt.str.match(r'^-?\d+\.\d{1,2}$'),
    amt.str.match(r'^-?\d+\.\d{3}$'),
    amt.str.match(r'^-?\d+$'),
]
labels = ['G2_comma', 'G1_normal', 'G3_three_dec', 'G4_no_dec']

df['AMT_GROUP'] = np.select(conditions, labels, default='OTHER')
df['AMT_GROUP'].value_counts()

,count
AMT_GROUP,
G1_normal,1680
G4_no_dec,583
G3_three_dec,25
G2_comma,8


In [32]:
g1 = df['AMT_GROUP'] == 'G1_normal'

amount   = pd.to_numeric(df.loc[g1, 'TXN_AMOUNT'])
expected = df.loc[g1, 'BILLING_AMOUNT'] / df.loc[g1, 'FX_RATE']

diff = (amount - expected).abs()
print('Rows that do not match:', (diff > 0.05).sum())

Rows that do not match: 58


In [33]:
def fix_comma(s):
    if ',' in s and '.' in s:
        if s.rfind(',') > s.rfind('.'):
            return s.replace('.', '').replace(',', '.')
        else:
            return s.replace(',', '')
    return s.replace(',', '.')

g2 = df['AMT_GROUP'] == 'G2_comma'

df.loc[g2, 'TXN_AMOUNT'] = df.loc[g2, 'TXN_AMOUNT'].apply(fix_comma)



df['TXN_AMOUNT'] = pd.to_numeric(df['TXN_AMOUNT'], errors='coerce')
print('NaN:', df['TXN_AMOUNT'].isna().sum())
print(df['TXN_AMOUNT'].dtype)

NaN: 0
float64


In [34]:
df = df.drop(columns=['new_ta'])

In [35]:
df.head()

,ACCOUNT_ID,TXN_ID,SETTLE_DATE,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,...,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,set_date,SETTLE_dt,SETTLE_DAYS,SETTLE_IMPUTED,AMT_GROUP
0,716075454,4700058,2022-03-13,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,...,Purchase,PDYR05FW,0MGT4L,Match,2022-03-10 00:51:36,9999-99-99,2022-03-13,3,False,G1_normal
1,551324309,4700106,10/07/2022,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,...,Purchase,D8TG2V7K,RZF979,Match,2022-07-09 17:49:00,99/99/9999,2022-07-10,1,False,G1_normal
2,749053287,4700140,16/09/2022,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,...,Purchase,00000000,326LGX,Match,2022-09-14 00:09:34,99/99/9999,2022-09-16,2,False,G1_normal
3,671284639,4700183,23-Apr-22,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,...,Purchase,00000000,8TRUC6,Match,2022-04-22 22:52:00,99-Apr-99,2022-04-23,1,False,G1_normal
4,649925833,4700220,2022-06-25,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,...,Purchase,PAXKQGM8,USH2SB,Match,2022-06-24 21:19:59,9999-99-99,2022-06-25,1,False,G1_normal


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2296 entries, 0 to 2295
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ACCOUNT_ID        2296 non-null   int64         
 1   TXN_ID            2296 non-null   int64         
 2   SETTLE_DATE       2293 non-null   object        
 3   TXN_TYPE          2296 non-null   object        
 4   TXN_AMOUNT        2296 non-null   float64       
 5   TXN_CCY           2296 non-null   object        
 6   BILLING_AMOUNT    2296 non-null   float64       
 7   BILLING_CURRENCY  2296 non-null   object        
 8   FX_RATE           2296 non-null   float64       
 9   MERCHANT_NAME     2296 non-null   object        
 10  MCC_CODE          2296 non-null   int64         
 11  MERCHANT_COUNTRY  2296 non-null   object        
 12  MERCHANT_CITY     2229 non-null   object        
 13  PROCESSING_CODE   2296 non-null   int64         
 14  PROCESSING_TYPE   2296 n

In [37]:
df['TXN_CCY'].value_counts()

,count
TXN_CCY,
USD,1111
LBP,481
EUR,245
GBP,154
AED,127
SAR,48
JPY,25
QAR,16
JOD,15


In [38]:
raw = pd.read_excel(path)

orig = raw['TXN_AMOUNT'].astype(str)

odd = orig.str.contains(r'[^0-9.,-]', regex=True)
print('Rows with odd characters:', odd.sum())


compare = pd.DataFrame({'before': orig[odd], 'after': df.loc[odd, 'TXN_AMOUNT']})
print(compare.head(30))

Rows with odd characters: 7
            before       after
130       (808.41)      808.41
328      (1458.94)     1458.94
380   (2326526.00)  2326526.00
558       (265.75)      265.75
1120       (75.88)       75.88
1620   (317027.00)   317027.00
2250       (19.67)       19.67


In [39]:
df.loc[odd, 'TXN_AMOUNT'] = -df.loc[odd, 'TXN_AMOUNT']
print(df.loc[odd, 'TXN_AMOUNT'])

130        -808.41
328       -1458.94
380    -2326526.00
558        -265.75
1120        -75.88
1620    -317027.00
2250        -19.67
Name: TXN_AMOUNT, dtype: float64


In [40]:

print(pd.crosstab(df['AMT_GROUP'], df['TXN_CCY']))


expected = df['BILLING_AMOUNT'] / df['FX_RATE']
df['AMT_RATIO'] = (df['TXN_AMOUNT'] / expected).round(2)

def ratio_box(r):
    if pd.isna(r):            return 'unknown'
    if 0.97 <= r <= 1.03:     return 'match'
    if -1.03 <= r <= -0.97:   return 'sign_flipped'
    if 95 <= r <= 105:        return 'x100'
    if 0.0095 <= r <= 0.0105: return 'div100'
    return 'other'

df['AMT_CHECK'] = df['AMT_RATIO'].apply(ratio_box)
print(df['AMT_CHECK'].value_counts())

wrong_sign = ((df['TXN_TYPE'] == 'Settled') & (df['TXN_AMOUNT'] > 0)) | \
             ((df['TXN_TYPE'] == 'Refund')  & (df['TXN_AMOUNT'] < 0))
print('Wrong sign vs TXN_TYPE:', wrong_sign.sum())

TXN_CCY       AED  AUD  CAD  CHF  EGP  EUR  GBP  JOD  JPY  KWD  LBP  QAR  SAR  \
AMT_GROUP                                                                       
G1_normal     124    7    5    6   14  242  147    3    0    1    2   16   48   
G2_comma        1    0    0    0    0    0    2    0    0    0    3    0    0   
G3_three_dec    0    0    0    0    0    0    0   12    0   13    0    0    0   
G4_no_dec       2    0    0    1    1    3    5    0   25    0  476    0    0   

TXN_CCY       SEK  TRY   USD  
AMT_GROUP                     
G1_normal      12   14  1039  
G2_comma        0    0     2  
G3_three_dec    0    0     0  
G4_no_dec       0    0    70  
AMT_CHECK
match           2241
x100              23
other             19
sign_flipped      13
Name: count, dtype: int64
Wrong sign vs TXN_TYPE: 13


In [41]:

same = wrong_sign & (df['AMT_CHECK'] == 'sign_flipped')
print('Same rows:', same.sum())


g4_other = (df['AMT_GROUP'] == 'G4_no_dec') & ~df['TXN_CCY'].isin(['LBP', 'JPY'])
print(df.loc[g4_other, 'AMT_CHECK'].value_counts())

Same rows: 13
AMT_CHECK
match    81
x100      1
Name: count, dtype: int64


In [42]:
df.loc[same, 'TXN_AMOUNT'] = -df.loc[same, 'TXN_AMOUNT']

In [43]:
df.head()

,ACCOUNT_ID,TXN_ID,SETTLE_DATE,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,...,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,set_date,SETTLE_dt,SETTLE_DAYS,SETTLE_IMPUTED,AMT_GROUP,AMT_RATIO,AMT_CHECK
0,716075454,4700058,2022-03-13,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,...,0MGT4L,Match,2022-03-10 00:51:36,9999-99-99,2022-03-13,3,False,G1_normal,1.0,match
1,551324309,4700106,10/07/2022,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,...,RZF979,Match,2022-07-09 17:49:00,99/99/9999,2022-07-10,1,False,G1_normal,1.0,match
2,749053287,4700140,16/09/2022,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,...,326LGX,Match,2022-09-14 00:09:34,99/99/9999,2022-09-16,2,False,G1_normal,1.0,match
3,671284639,4700183,23-Apr-22,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,...,8TRUC6,Match,2022-04-22 22:52:00,99-Apr-99,2022-04-23,1,False,G1_normal,1.0,match
4,649925833,4700220,2022-06-25,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,...,USH2SB,Match,2022-06-24 21:19:59,9999-99-99,2022-06-25,1,False,G1_normal,1.0,match


In [44]:
print(df.dtypes)

ACCOUNT_ID                   int64
TXN_ID                       int64
SETTLE_DATE                 object
TXN_TYPE                    object
TXN_AMOUNT                 float64
TXN_CCY                     object
BILLING_AMOUNT             float64
BILLING_CURRENCY            object
FX_RATE                    float64
MERCHANT_NAME               object
MCC_CODE                     int64
MERCHANT_COUNTRY            object
MERCHANT_CITY               object
PROCESSING_CODE              int64
PROCESSING_TYPE             object
TERMINAL_ID                 object
AUTH_CODE                   object
MATCHES_STATUS              object
TXN_DATE_TIME       datetime64[ns]
set_date                    object
SETTLE_dt           datetime64[ns]
SETTLE_DAYS                  int64
SETTLE_IMPUTED                bool
AMT_GROUP                   object
AMT_RATIO                  float64
AMT_CHECK                   object
dtype: object


In [45]:
df = df.drop(columns=['set_date', 'AMT_GROUP', 'AMT_RATIO', 'SETTLE_DATE'], errors='ignore')
df = df.rename(columns={'SETTLE_dt': 'SETTLE_DATE'})
print(df.columns)

Index(['ACCOUNT_ID', 'TXN_ID', 'TXN_TYPE', 'TXN_AMOUNT', 'TXN_CCY',
       'BILLING_AMOUNT', 'BILLING_CURRENCY', 'FX_RATE', 'MERCHANT_NAME',
       'MCC_CODE', 'MERCHANT_COUNTRY', 'MERCHANT_CITY', 'PROCESSING_CODE',
       'PROCESSING_TYPE', 'TERMINAL_ID', 'AUTH_CODE', 'MATCHES_STATUS',
       'TXN_DATE_TIME', 'SETTLE_DATE', 'SETTLE_DAYS', 'SETTLE_IMPUTED',
       'AMT_CHECK'],
      dtype='object')


In [46]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,MERCHANT_NAME,MCC_CODE,...,PROCESSING_CODE,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,AMT_CHECK
0,716075454,4700058,Settled,-104.39,USD,-104.39,USD,1.000000,BOU KHALIL MKT,5411,...,0,Purchase,PDYR05FW,0MGT4L,Match,2022-03-10 00:51:36,2022-03-13,3,False,match
1,551324309,4700106,Settled,-162.99,USD,-162.99,USD,1.000000,IZ *BP FUEL,5999,...,0,Purchase,D8TG2V7K,RZF979,Match,2022-07-09 17:49:00,2022-07-10,1,False,match
2,749053287,4700140,Settled,-480.79,USD,-480.79,USD,1.000000,HRTZ RENTAL,7538,...,0,Purchase,00000000,326LGX,Match,2022-09-14 00:09:34,2022-09-16,2,False,match
3,671284639,4700183,Settled,-489.74,USD,-489.74,USD,1.000000,HRTZ RENTAL,7538,...,0,Purchase,00000000,8TRUC6,Match,2022-04-22 22:52:00,2022-04-23,1,False,match
4,649925833,4700220,Settled,-58.09,AED,-15.54,USD,0.267538,DUBAI MN /ufw,4111,...,0,Purchase,PAXKQGM8,USH2SB,Match,2022-06-24 21:19:59,2022-06-25,1,False,match


In [47]:
df['AMT_CHECK'].value_counts()

,count
AMT_CHECK,
match,2241
x100,23
other,19
sign_flipped,13


In [48]:
x100 = df['AMT_CHECK'] == 'x100'
print(df.loc[x100, ['TXN_AMOUNT', 'FX_RATE', 'BILLING_AMOUNT']].head())


df['BILLING_FIXED'] = x100
df.loc[x100, 'BILLING_AMOUNT'] = (df.loc[x100, 'TXN_AMOUNT'] * df.loc[x100, 'FX_RATE']).round(2)

print(df.loc[x100, ['TXN_AMOUNT', 'FX_RATE', 'BILLING_AMOUNT']].head())

     TXN_AMOUNT   FX_RATE  BILLING_AMOUNT
48      -154.77  1.000000           -1.55
115      -36.35  1.000000           -0.36
164     -188.55  1.086101           -2.05
302    -3375.10  1.000000          -33.75
571 -3851979.00  0.000663          -25.55
     TXN_AMOUNT   FX_RATE  BILLING_AMOUNT
48      -154.77  1.000000         -154.77
115      -36.35  1.000000          -36.35
164     -188.55  1.086101         -204.78
302    -3375.10  1.000000        -3375.10
571 -3851979.00  0.000663        -2555.21


In [49]:
expected = df['BILLING_AMOUNT'] / df['FX_RATE']
ratio = (df['TXN_AMOUNT'] / expected).round(2)
print(ratio.apply(ratio_box).value_counts())

match    2277
other      19
Name: count, dtype: int64


In [50]:
df['BILLING_CURRENCY'].value_counts()

,count
BILLING_CURRENCY,
USD,2296


In [51]:
df = df.drop(columns=['BILLING_CURRENCY'])

In [52]:

good = ratio.apply(ratio_box) == 'match'
usual_fx = df.loc[good].groupby('TXN_CCY')['FX_RATE'].median()


other = ratio.apply(ratio_box) == 'other'
look = df.loc[other, ['TXN_CCY', 'TXN_AMOUNT', 'FX_RATE', 'BILLING_AMOUNT']].copy()
look['USUAL_FX'] = look['TXN_CCY'].map(usual_fx)
look['RATIO'] = ratio[other]
print(look)
df['AMT_MISMATCH'] = other
print(df['AMT_MISMATCH'].sum())



     TXN_CCY  TXN_AMOUNT   FX_RATE  BILLING_AMOUNT  USUAL_FX  RATIO
107      USD     -154.26  1.000000         -174.13  1.000000   0.89
274      AED     -670.57  0.266698         -204.51  0.271927   0.87
319      LBP -9741141.00  0.000029        -6461.79  0.000037   0.04
434      USD      -19.99  1.000000          -21.39  1.000000   0.93
523      USD     -194.83  1.000000         -222.07  1.000000   0.88
560      USD      -56.65  1.000000          -61.64  1.000000   0.92
714      USD      -19.99  1.000000          -21.99  1.000000   0.91
743      GBP      -12.51  1.246303          -17.74  1.266205   0.88
824      AED    -1159.09  0.270696         -339.08  0.271927   0.93
937      USD     -212.05  1.000000         -225.73  1.000000   0.94
988      EUR     -145.77  1.086163         -166.25  1.083733   0.95
1551     USD     -263.62  1.000000         -300.05  1.000000   0.88
1579     USD      -65.27  1.000000          -69.90  1.000000   0.93
1614     AED     -185.28  0.269866          -56.

In [53]:
df=df.drop('AMT_CHECK',axis=1)

In [54]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MERCHANT_NAME,MCC_CODE,MERCHANT_COUNTRY,...,PROCESSING_TYPE,TERMINAL_ID,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,BOU KHALIL MKT,5411,LB,...,Purchase,PDYR05FW,0MGT4L,Match,2022-03-10 00:51:36,2022-03-13,3,False,False,False
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,IZ *BP FUEL,5999,GB,...,Purchase,D8TG2V7K,RZF979,Match,2022-07-09 17:49:00,2022-07-10,1,False,False,False
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,HRTZ RENTAL,7538,US,...,Purchase,00000000,326LGX,Match,2022-09-14 00:09:34,2022-09-16,2,False,False,False
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,HRTZ RENTAL,7538,US,...,Purchase,00000000,8TRUC6,Match,2022-04-22 22:52:00,2022-04-23,1,False,False,False
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,DUBAI MN /ufw,4111,AE,...,Purchase,PAXKQGM8,USH2SB,Match,2022-06-24 21:19:59,2022-06-25,1,False,False,False


In [55]:
names = df['MERCHANT_NAME']

print('Different names:', names.nunique())
print(names.value_counts().head(30))

odd = names[names.str.contains(r'[^A-Za-z0-9 ]', regex=True)]
print(odd.value_counts().head(30))

Different names: 1678
MERCHANT_NAME
SKRILL.COM                7
SAXO B                    7
DEUTSCHE BAHN BR 75       6
STCSAUDI.COM              6
MEDGULF INSURANCE         6
CARREFOUR F               6
META A                    6
ZARA BR 441               5
ZAATAR W ZEIT #97         5
DUOLINGO.COM              5
LAU BYBLOS                5
BNP PARIBAS ATM BR 369    5
KAAK AS                   5
SANTANDER ATM BR 466      5
VSTPRNT                   5
TST* NOON COM             5
COINBASE.COM              5
HRTZ RENTAL               5
SPNNYS EXPRESS            5
FCS LEBANON               5
BRT WATER EST             5
STARBUCKS                 5
AUDIBLE                   5
ELECTRO CHAABAN #96       5
ARAMEX L                  4
THE BODY SHOP #1          4
IZ *OPENAI                4
DUBAI METRO NOL           4
STRIPEPAYOUT.COM          4
CAREEM                    4
Name: count, dtype: int64
MERCHANT_NAME
SKRILL.COM                  7
STCSAUDI.COM                6
TST* NOON COM           

In [56]:
df['MERCHANT_CLEAN']=df['MERCHANT_NAME'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)
df['MERCHANT_CLEAN'].nunique()



1310

In [57]:
KNOWN = ['SQ', 'IZ', 'TST', 'WPY', 'SP', 'PP', 'PAYPAL', 'GOOGLE']

def split_prefix(name):
    if '*' in name:
        before, after = name.split('*', 1)
        before, after = before.strip(), after.strip()
        if before in KNOWN:
            return before, after
        else:
            return 'DIRECT', before
    return 'DIRECT', name

print(split_prefix('PP*AMAZON MARKETPLACE'))
print(split_prefix('TOTERS *0UBA'))
print(split_prefix('STARBUCKS'))

('PP', 'AMAZON MARKETPLACE')
('DIRECT', 'TOTERS')
('DIRECT', 'STARBUCKS')


In [58]:
df['MERCHANT_CLEAN'] = df['MERCHANT_NAME'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)
df[['PAYMENT_VIA', 'MERCHANT_CLEAN']] = df['MERCHANT_CLEAN'].apply(split_prefix).apply(pd.Series)
df['PAYMENT_VIA'] = df['PAYMENT_VIA'].replace({'PP': 'PAYPAL'})

print(df['PAYMENT_VIA'].value_counts())
print('Different names:', df['MERCHANT_CLEAN'].nunique())

PAYMENT_VIA
DIRECT    1889
SQ          83
PAYPAL      75
IZ          72
TST         55
WPY         45
SP          42
GOOGLE      35
Name: count, dtype: int64
Different names: 1098


In [59]:
m = df['MERCHANT_CLEAN']
print(m[m.str.contains('/')].value_counts().head(20))
print(m[m.str.contains(r'\bBR\b')].value_counts().head(20))

MERCHANT_CLEAN
DUBAI MN /UFW           1
SNCF CONNECT /TUZ       1
VUE CINEMAS #81 /BCF    1
AUTOZONE /LZX           1
VODAFONE EGYPT /XVI     1
NOON COM /CEC           1
RPSL /ZPJ               1
PLYSTTN STORE /GHD      1
ALDREES PETROL /UBG     1
BTQ ELIE SAAB /QCZ      1
MEDIA MARKT #59 /PPO    1
NINTENDO E /FFR         1
SUBWAY BR 798 /OAQ      1
REPSOL /XZJ             1
CRM /GHM                1
MEZYAN BR 440 /CMW      1
BTS PHARMACY /HFF       1
BINANCE.COM /RGK        1
STRBCKS /BWW            1
XBOX L /HVL             1
Name: count, dtype: int64
MERCHANT_CLEAN
FEDEX BR 762                 8
DEUTSCHE BAHN BR 75          7
SANTANDER ATM BR 466         7
ZARA BR 441                  6
NHS PRESCRIPTION BR 855      6
MEZYANE BR 233               5
VIRGIN MOBILE KSA BR 481     5
BNP PARIBAS ATM BR 369       5
UNIQLO BR 128                5
BEIRUT ROASTERY BR 571       5
BARNES AND NOBLE BR 632      5
TOTAL LEBANON BR 557         4
ELECTRICITE DU LIBAN BR      4
CHATEAU KSARA BR 870 

In [60]:
m = df['MERCHANT_CLEAN']
m = m.str.replace(r'\s*/\w+$', '', regex=True)
m = m.str.replace(r'\s*#\d+$', '', regex=True)
m = m.str.replace(r'\s+BR(\s*\d+)?$', '', regex=True)
df['MERCHANT_CLEAN'] = m.str.strip()

print('Different names:', df['MERCHANT_CLEAN'].nunique())

Different names: 782


In [61]:
first = df['MERCHANT_CLEAN'].str.split().str[0]
groups = df.groupby(first)['MERCHANT_CLEAN'].unique()
multi = groups[groups.str.len() > 1]

print('Groups with variants:', len(multi))
for word, names in multi.head(30).items():
    print(word, '→', list(names))

Groups with variants: 202
ABC → ['ABC VERDUN *87YA', 'ABC C', 'ABC VERDUN', 'ABC V', 'ABC CONSULTING', 'ABC VERDUN 13475']
ABOUCHACRA → ['ABOUCHACRA', 'ABOUCHACRA #43 81378']
ADB → ['ADB CREATIVE CLOUD 62524', 'ADB CREATIVE CLOUD']
ADNOC → ['ADNOC DISTRIBUTION', 'ADNOC DISTR', 'ADNOC D']
ALDI → ['ALDI SUD', 'ALDI SUD BR 722 82038', 'ALDI SUD BR 722 45289', 'ALDI SUD BR 722 15126', 'ALDI S']
ALDREES → ['ALDREES PETROL', 'ALDREES P']
ALFA → ['ALFA TELECOM', 'ALFA TELECOM *KX8Y']
ALIEXPRESS → ['ALIEXPRESS 96357', 'ALIEXPRESS']
ALLNZ → ['ALLNZ DIRECT', 'ALLNZ DIRECT 32483']
ALLO → ['ALLO TAXI 4106', 'ALLO T', 'ALLO TAXI']
AMAZON → ['AMAZON M', 'AMAZON MARKETPLACE']
AMC → ['AMC T', 'AMC THEATRES', 'AMC T 55460']
AMERICAN → ['AMERICAN UNIVERSITY', 'AMERICAN UNIV', 'AMERICAN U']
ANGHAMI → ['ANGHAMI 17294', 'ANGHAMI']
APPLE → ['APPLE STORE 88741', 'APPLE MUSIC', 'APPLE STR', 'APPLE STORE', 'APPLE M', 'APPLE STORE #47 69120']
ARAMEX → ['ARAMEX LEBANON', 'ARAMEX L', 'ARAMEX LEBANON #26 76020']
A

In [62]:
import re

def strip_junk(s):
    for _ in range(2):
        s = re.sub(r'\s*\*\w+$', '', s)
        s = re.sub(r'\s+\d{4,}$', '', s)
        s = re.sub(r'\s*#\d+$', '', s)
        s = re.sub(r'\s+BR(\s*\d+)?$', '', s)
        s = re.sub(r'\s*/\w+$', '', s)
    return s.strip()

print(strip_junk('ALDI SUD BR 722 82038'))
print(strip_junk('ABOUCHACRA #43 81378'))

df['MERCHANT_CLEAN'] = df['MERCHANT_CLEAN'].apply(strip_junk)
print('Different names:', df['MERCHANT_CLEAN'].nunique())

ALDI SUD
ABOUCHACRA
Different names: 629


In [63]:
names = df['MERCHANT_CLEAN'].unique()
mapping = {}

for n in names:
    if len(n.split()) < 2:                      # skip single words (ZARA ≠ ZARA HOME)
        continue
    cands = [c for c in names if c != n and c.startswith(n)]
    if not cands:
        continue
    longest = max(cands, key=len)
    if all(longest.startswith(c) for c in cands):   # all point to ONE full name
        mapping[n] = longest

print('Names to map:', len(mapping))
for short, full in list(mapping.items())[:40]:
    print(short, '→', full)

Names to map: 93
OOREDOO Q → OOREDOO QATAR
JUMEIRAH BEACH HOTEL → JUMEIRAH BEACH HOTEL B
CARREFOUR F → CARREFOUR FRANCE
YOUTUBE PREM → YOUTUBE PREMIUM
BYBLOS BA → BYBLOS BANK ATM
ZOOM V → ZOOM VIDEO
MAJESTIC W → MAJESTIC WINE
MADE C → MADE COM
KRAKEN E → KRAKEN EXCHANGE
SAXO B → SAXO BANK
QATAR A → QATAR AIRWAYS
DHL E → DHL EXPRESS
PHOENICIA H → PHOENICIA HOTEL
BEIRUT R → BEIRUT ROASTERY
FAHED S → FAHED SPRMKT
CITY E → CITY ELECTRONICS
ETSY S → ETSY SELLER
NINTENDO E → NINTENDO ESHOP
REGUS O → REGUS OFFICES
LULU H → LULU HYPERMARKET
UBER E → UBER EATS
USJ B → USJ BEIRUT
TOTAL L → TOTAL LEBANON
QUEST D → QUEST DIAGNOSTICS
AMAZON M → AMAZON MARKETPLACE
SLACK TECH → SLACK TECHNOLOGIES
KUWAIT A → KUWAIT AIRWAYS
RYMCO SP → RYMCO SPARE PARTS
SAFEWAY J → SAFEWAY JORDAN
YOUTUBE P → YOUTUBE PREMIUM
JARIR B → JARIR BOOKSTORE
XBOX L → XBOX LIVE
MEPHICO P → MEPHICO PHARMACY
CARREFOUR E → CARREFOUR EGYPT
ABC C → ABC CONSULTING
DISNEY P → DISNEY PLUS
META A → META ADS
PURE G → PURE GYM
COSTA C → COS

In [64]:

bad = {s: f for s, f in mapping.items() if len(f.split()[-1]) == 1}
for s, f in bad.items():
    del mapping[s]
    mapping[f] = s
print('Reversed:', bad)

for short, full in list(mapping.items())[40:]:
    print(short, '→', full)

Reversed: {'JUMEIRAH BEACH HOTEL': 'JUMEIRAH BEACH HOTEL B'}
ENOC S → ENOC STATION
BEAUTY LOUNGE H → BEAUTY LOUNGE HAMRA
BOLT L → BOLT LEBANON
ARAMEX L → ARAMEX LEBANON
BLOM BA → BLOM BANK ATM
BOB F → BOB FINANCE
FACES L → FACES LEBANON
TOTAL W → TOTAL WINE
STEAM G → STEAM GAMES
AXA INS → AXA INSURANCE
ADNOC DISTR → ADNOC DISTRIBUTION
AMC T → AMC THEATRES
ALLO T → ALLO TAXI
SPINNEYS E → SPINNEYS EXPRESS
MARRIOTT H → MARRIOTT HOTELS
NOTION L → NOTION LABS
SLACK T → SLACK TECHNOLOGIES
AMERICAN UNIV → AMERICAN UNIVERSITY
TRADER J → TRADER JOES
CHASE BA → CHASE BANK ATM
DU T → DU TELECOM
ALDREES P → ALDREES PETROL
SAUDIA A → SAUDIA AIRLINES
ABC V → ABC VERDUN
SAMS C → SAMS CLUB
ADNOC D → ADNOC DISTRIBUTION
BURGER K → BURGER KING
MICROSOFT 3 → MICROSOFT 365
STAPLES O → STAPLES OFFICE
KINDLE S → KINDLE STR
LINKEDIN P → LINKEDIN PREMIUM
WISE T → WISE TRANSFER
SHOPIFY B → SHOPIFY BILLING
EPIC G → EPIC GAMES
JIFFY L → JIFFY LUBE
BANKERS A → BANKERS ASSURANCE
EMIRATES A → EMIRATES AIRLINE
AMERIC

In [65]:
df['MERCHANT_CLEAN'] = df['MERCHANT_CLEAN'].replace(mapping)
print('Different names:', df['MERCHANT_CLEAN'].nunique())

Different names: 536


In [66]:
m = df['MERCHANT_CLEAN']
no_vowels = m[~m.str.contains('[AEIOU]')]
print(no_vowels.value_counts().head(40))

MERCHANT_CLEAN
VSTPRNT     8
WTRSTNS     6
STRBCKS     5
CVS PHCY    5
WYFR        5
BRBR        4
SPNNYS      4
MNPRX       4
MCDNLDS     4
KFC         3
RVLT        3
MNYGRM      3
TSC S       3
SHN         3
CNV         3
TTRS        3
MNPG        3
DR NH       2
KLRN        2
GTHB        2
RYNR        2
TRNTL       2
NTFLX       2
CRM         2
HLFRDS      2
RPSL        1
FLPKRT      1
SQRSPC      1
CNBS        1
FLYDB       1
FVRR        1
DLVR        1
CRSR        1
DGTLCN      1
GRMMRLY     1
CVS P       1
SHRF DG     1
WTRS        1
MLCHMP      1
BNNC        1
Name: count, dtype: int64


In [67]:
typo_map = {
    'VSTPRNT': 'VISTAPRINT',      'WTRSTNS': 'WATERSTONES',
    'STRBCKS': 'STARBUCKS',       'CVS PHCY': 'CVS PHARMACY',
    'CVS P': 'CVS PHARMACY',      'WYFR': 'WAYFAIR',
    'SPNNYS': 'SPINNEYS',         'MNPRX': 'MONOPRIX',
    'MCDNLDS': 'MCDONALDS',       'RVLT': 'REVOLUT',
    'MNYGRM': 'MONEYGRAM',        'SHN': 'SHEIN',
    'CNV': 'CANVA',               'TTRS': 'TOTERS',
    'KLRN': 'KLARNA',             'GTHB': 'GITHUB',
    'RYNR': 'RYANAIR',            'NTFLX': 'NETFLIX',
    'HLFRDS': 'HALFORDS',         'RPSL': 'REPSOL',
    'FLPKRT': 'FLIPKART',         'SQRSPC': 'SQUARESPACE',
    'CNBS': 'COINBASE',           'FLYDB': 'FLYDUBAI',
    'FVRR': 'FIVERR',             'DLVR': 'DELIVEROO',
    'CRSR': 'COURSERA',           'DGTLCN': 'DIGITALOCEAN',
    'GRMMRLY': 'GRAMMARLY',       'SHRF DG': 'SHARAF DG',
    'MLCHMP': 'MAILCHIMP',        'BNNC': 'BINANCE',
    'DR NH': 'DR NADIA HADDAD',
}

m = df['MERCHANT_CLEAN']
existing = set(m.unique())


missing = [t for t in set(typo_map.values()) if t not in existing]
for t in missing:
    similar = [n for n in existing if t[:5] in n]
    print(t, '→ found in data:', similar)

WAYFAIR → found in data: ['WAYFAIR.COM']


In [68]:
typo_map['WYFR'] = 'WAYFAIR.COM'
df['MERCHANT_CLEAN'] = df['MERCHANT_CLEAN'].replace(typo_map)
print('Different names:', df['MERCHANT_CLEAN'].nunique())

Different names: 503


In [69]:
df['MCC_CODE'].value_counts()

,count
MCC_CODE,
5999,299
5411,165
7372,108
4829,98
5812,95
4511,82
5815,80
5541,79
6011,71


In [70]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MERCHANT_NAME,MCC_CODE,MERCHANT_COUNTRY,...,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,MERCHANT_CLEAN,PAYMENT_VIA
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,BOU KHALIL MKT,5411,LB,...,0MGT4L,Match,2022-03-10 00:51:36,2022-03-13,3,False,False,False,BOU KHALIL MKT,DIRECT
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,IZ *BP FUEL,5999,GB,...,RZF979,Match,2022-07-09 17:49:00,2022-07-10,1,False,False,False,BP FUEL,IZ
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,HRTZ RENTAL,7538,US,...,326LGX,Match,2022-09-14 00:09:34,2022-09-16,2,False,False,False,HRTZ RENTAL,DIRECT
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,HRTZ RENTAL,7538,US,...,8TRUC6,Match,2022-04-22 22:52:00,2022-04-23,1,False,False,False,HRTZ RENTAL,DIRECT
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,DUBAI MN /ufw,4111,AE,...,USH2SB,Match,2022-06-24 21:19:59,2022-06-25,1,False,False,False,DUBAI MN,DIRECT


In [71]:
mcc_per_merchant = df.groupby('MERCHANT_CLEAN')['MCC_CODE'].nunique()
mixed = mcc_per_merchant[mcc_per_merchant > 1]
print('Merchants with 2+ different MCCs:', len(mixed))


for name in mixed.index[:15]:
    print(name, '→', df.loc[df['MERCHANT_CLEAN'] == name, 'MCC_CODE'].value_counts().to_dict())

Merchants with 2+ different MCCs: 143
ABC CONSULTING → {7399: 3, 5999: 1}
ABC VERDUN → {5812: 4, 5999: 4}
ABOUCHACRA → {5712: 4, 5999: 3}
ADIDAS → {5651: 3, 5999: 2}
ADNOC DISTRIBUTION → {5541: 7, 5999: 1}
ADOBE CREATIVE CLOUD → {7372: 3, 5999: 2}
AL MEERA DOHA → {5411: 3, 5812: 2}
ALDI SUD → {5411: 9, 5999: 1}
ALFA TELECOM → {4814: 5, 5999: 1}
ALLO TAXI → {4121: 6, 5999: 2}
ANGHAMI → {4899: 3, 5812: 2}
APPLE STORE → {5812: 3, 5732: 2}
AUB MEDICAL CENTER → {8062: 7, 5812: 3}
AVIS CAR RENTAL → {7538: 4, 5817: 2}
AXA INSURANCE → {6300: 5, 5817: 2}


In [72]:
df = df.drop(columns=['MERCHANT_NAME'])
df = df.rename(columns={'MERCHANT_CLEAN': 'MERCHANT_NAME'})
print(df.columns)

Index(['ACCOUNT_ID', 'TXN_ID', 'TXN_TYPE', 'TXN_AMOUNT', 'TXN_CCY',
       'BILLING_AMOUNT', 'FX_RATE', 'MCC_CODE', 'MERCHANT_COUNTRY',
       'MERCHANT_CITY', 'PROCESSING_CODE', 'PROCESSING_TYPE', 'TERMINAL_ID',
       'AUTH_CODE', 'MATCHES_STATUS', 'TXN_DATE_TIME', 'SETTLE_DATE',
       'SETTLE_DAYS', 'SETTLE_IMPUTED', 'BILLING_FIXED', 'AMT_MISMATCH',
       'MERCHANT_NAME', 'PAYMENT_VIA'],
      dtype='object')


In [73]:
df['MERCHANT_COUNTRY'].value_counts()

,count
MERCHANT_COUNTRY,
US,749
LB,581
GB,223
AE,171
FR,73
DE,68
SA,61
TR,60
JP,30


In [74]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,...,AUTH_CODE,MATCHES_STATUS,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,MERCHANT_NAME,PAYMENT_VIA
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,5411,LB,ZAHLE,...,0MGT4L,Match,2022-03-10 00:51:36,2022-03-13,3,False,False,False,BOU KHALIL MKT,DIRECT
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,5999,GB,LONDON,...,RZF979,Match,2022-07-09 17:49:00,2022-07-10,1,False,False,False,BP FUEL,IZ
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,7538,US,ECOM,...,326LGX,Match,2022-09-14 00:09:34,2022-09-16,2,False,False,False,HRTZ RENTAL,DIRECT
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,7538,US,E-COMMERCE,...,8TRUC6,Match,2022-04-22 22:52:00,2022-04-23,1,False,False,False,HRTZ RENTAL,DIRECT
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,4111,AE,DUBAYY,...,USH2SB,Match,2022-06-24 21:19:59,2022-06-25,1,False,False,False,DUBAI MN,DIRECT


In [75]:
df['MERCHANT_CITY'].value_counts()

,count
MERCHANT_CITY,
INTERNET,295
E-COMMERCE,167
ECOM,152
LONDON,116
BEIRUT LB,88
...,...
tokyo,1
saida,1
ann arbor,1


In [76]:
df['MERCHANT_CITY']=df['MERCHANT_CITY'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)

In [77]:
df['MERCHANT_CITY'].value_counts()

,count
MERCHANT_CITY,
INTERNET,305
E-COMMERCE,173
ECOM,158
LONDON,121
BEIRUT LB,91
...,...
CAPE TOWN,2
BROOKLYN,2
CARY,2


In [78]:
df['MERCHANT_CITY'] = df['MERCHANT_CITY'].replace({
    'INTERNET': 'ONLINE',
    'E-COMMERCE': 'ONLINE',
    'ECOM': 'ONLINE',
})
print(df['MERCHANT_CITY'].value_counts().head())

MERCHANT_CITY
ONLINE       636
LONDON       121
BEIRUT LB     91
BEIRUT        87
BEYROUTH      82
Name: count, dtype: int64


In [79]:
df['MERCHANT_CITY'] = df['MERCHANT_CITY'].replace({
    'BEIRUT LB': 'BEIRUT',
    'BEYRUT': 'BEIRUT',
    'BEYROUTH': 'BEIRUT',
})
print(df['MERCHANT_CITY'].value_counts())

MERCHANT_CITY
ONLINE       636
BEIRUT       339
LONDON       121
DUBAI         66
PARIS         59
            ... 
CAPE TOWN      2
BROOKLYN       2
CARY           2
MUNICH         2
DELFT          1
Name: count, Length: 122, dtype: int64


In [80]:
df['MERCHANT_CITY'] = df['MERCHANT_CITY'].replace({
    'DUBAI AE': 'DUBAI',

})
print(df['MERCHANT_CITY'].value_counts())

MERCHANT_CITY
ONLINE       636
BEIRUT       339
LONDON       121
DUBAI         66
PARIS         59
            ... 
CAPE TOWN      2
BROOKLYN       2
CARY           2
MUNICH         2
DELFT          1
Name: count, Length: 122, dtype: int64


In [81]:
c = df['MERCHANT_CITY'].dropna()
ends_code = c[c.str.contains(r' [A-Z]{2}$')]
print(ends_code.value_counts())

Series([], Name: count, dtype: int64)


In [82]:
print(sorted(df['MERCHANT_CITY'].dropna().unique()))

['ABU DHABI', 'ACHRAFIEH', 'AL HAMRA', 'AL KUWAYT', 'AL QAHIRAH', 'AMMAN', 'AMSTERDAM', 'ANN ARBOR', 'ASHRAFIEH', 'ASHRAFIYA', 'ATLANTA', 'AUSTIN', 'BAABDA', 'BANGALORE', 'BEAVERTON', 'BEIRUT', 'BELLEVUE', 'BENTONVILLE', 'BERLIN', 'BETHESDA', 'BONN', 'BOSTON', 'BRACKNELL', 'BROOKLYN', 'BUENA PARK', 'BUENOS AIRES', 'BURBANK', 'BYBLOS', 'CAIRO', 'CAPE TOWN', 'CARY', 'CHICAGO', 'COPENHAGEN', 'CUPERTINO', 'DALLAS', 'DEERFIELD', 'DELFT', 'DOHA', 'DUBAI', 'DUBAYY', 'DUBLIN', 'DUNSTABLE', 'ESTERO', 'FRAMINGHAM', 'FRANKFURT', 'GREENWICH', 'GUANGZHOU', 'HAMPTON', 'HAMRA', 'HANGZHOU', 'HERZOGENAURACH', 'HOUSTON', 'INGOLSTADT', 'ISSAQUAH', 'ISTANBOUL', 'ISTANBUL', 'JBAIL', 'JBEIL', 'JEDDAH', 'JOUNIE', 'JOUNIEH', 'JUNIYAH', 'KANSAS CITY', 'KUWAIT CITY', 'KYOTO', 'LAGOS', 'LEAWOOD', 'LEEDS', 'LILLE', 'LONDON', 'LOS GATOS', 'LOUISVILLE', 'MADRID', 'MEMPHIS', 'MENLO PARK', 'MIAMI', 'MILFORD', 'MONROVIA', 'MOUNTAIN VIEW', 'MULHEIM', 'MUNICH', 'NECKARSULM', 'NEW YORK', 'NEWARK', 'NOTTINGHAM', 'ONLINE',

In [83]:
city_map = {
    'DUBAYY': 'DUBAI',
    'ISTANBOUL': 'ISTANBUL',
    'AL QAHIRAH': 'CAIRO',
    'AL KUWAYT': 'KUWAIT CITY',
    'JBAIL': 'BYBLOS',        'JBEIL': 'BYBLOS',
    'JOUNIE': 'JOUNIEH',      'JUNIYAH': 'JOUNIEH',
    'SAYDA': 'SAIDA',         'SIDON': 'SAIDA',
    'TRABLOUS': 'TRIPOLI',    'TARABLUS': 'TRIPOLI',
    'ZAHLEH': 'ZAHLE',
    'ASHRAFIEH': 'ACHRAFIEH', 'ASHRAFIYA': 'ACHRAFIEH',
    'AL HAMRA': 'HAMRA',

    'ACHRAFIEH': 'BEIRUT',
    'HAMRA': 'BEIRUT',
}


trip = df['MERCHANT_CITY'].isin(['TRIPOLI', 'TRABLOUS', 'TARABLUS'])
print(df.loc[trip, 'MERCHANT_COUNTRY'].value_counts())

df['MERCHANT_CITY'] = df['MERCHANT_CITY'].replace(city_map)
print('Different cities:', df['MERCHANT_CITY'].nunique())

MERCHANT_COUNTRY
LB    33
TR     2
US     1
Name: count, dtype: int64
Different cities: 106


In [84]:
odd = trip & (df['MERCHANT_COUNTRY'] != 'LB')
print(df.loc[odd, ['MERCHANT_NAME', 'MERCHANT_CITY', 'MERCHANT_COUNTRY', 'TXN_CCY']])

        MERCHANT_NAME MERCHANT_CITY MERCHANT_COUNTRY TXN_CCY
433     BLOM BANK ATM       TRIPOLI               TR     USD
519  MEPHICO PHARMACY       TRIPOLI               US     LBP
685     BLOM BANK ATM       TRIPOLI               TR     LBP


In [85]:
df.loc[odd, 'MERCHANT_COUNTRY'] = 'LB'
cc = df.dropna(subset=['MERCHANT_CITY']).groupby('MERCHANT_CITY')['MERCHANT_COUNTRY'].nunique()
mixed = cc[(cc > 1) & (cc.index != 'ONLINE')]
print('Cities with 2+ countries:', len(mixed))
for city in mixed.index:
    rows = df['MERCHANT_CITY'] == city
    print(city, '→', df.loc[rows, 'MERCHANT_COUNTRY'].value_counts().to_dict())

Cities with 2+ countries: 43
ABU DHABI → {'AE': 18, 'GB': 2}
ACHRAFIEH → {'LB': 13, 'US': 2}
AMMAN → {'JO': 14, 'TR': 2, 'US': 2}
AMSTERDAM → {'NL': 6, 'US': 2}
ANN ARBOR → {'US': 5, 'TR': 1}
ATLANTA → {'US': 7, 'GB': 1}
BEIRUT → {'LB': 311, 'US': 26, 'TR': 5, 'GB': 4, 'IE': 2}
BERLIN → {'DE': 6, 'US': 4}
BETHESDA → {'US': 16, 'GB': 2}
BONN → {'DE': 4, 'US': 2}
BRACKNELL → {'GB': 3, 'TR': 2}
BUENOS AIRES → {'AR': 4, 'TR': 1}
BYBLOS → {'LB': 24, 'US': 1}
DALLAS → {'US': 8, 'IE': 1}
DOHA → {'QA': 10, 'US': 2, 'GB': 1}
DUBAI → {'AE': 105, 'US': 8, 'GB': 2}
DUNSTABLE → {'GB': 5, 'US': 1}
HAMRA → {'LB': 7, 'US': 1}
INGOLSTADT → {'DE': 9, 'TR': 1}
ISSAQUAH → {'US': 5, 'GB': 3}
JEDDAH → {'SA': 13, 'US': 3, 'GB': 1}
JOUNIEH → {'LB': 46, 'US': 2, 'TR': 1, 'IE': 1}
KUWAIT CITY → {'KW': 14, 'US': 3}
KYOTO → {'JP': 4, 'US': 1}
LEEDS → {'GB': 8, 'US': 1}
LONDON → {'GB': 112, 'US': 6, 'TR': 3}
MADRID → {'ES': 17, 'US': 2}
MIAMI → {'US': 4, 'TR': 1}
MULHEIM → {'DE': 8, 'US': 1}
NEW YORK → {'US': 25, 

In [86]:

main = df.groupby('MERCHANT_CITY')['MERCHANT_COUNTRY'].agg(lambda s: s.value_counts().index[0])
expected = df['MERCHANT_CITY'].map(main)


fix = expected.notna() & (df['MERCHANT_CITY'] != 'ONLINE') & (df['MERCHANT_COUNTRY'] != expected)

df['COUNTRY_FIXED'] = fix
df.loc[fix, 'MERCHANT_COUNTRY'] = expected[fix]
print('Rows fixed:', fix.sum())

Rows fixed: 144


In [87]:
miss = df['MERCHANT_CITY'].isna()
print(df.loc[miss, 'MERCHANT_NAME'].value_counts().head(15))

MERCHANT_NAME
SANTANDER ATM        3
STARBUCKS            2
DUOLINGO.COM         2
VERO MODA BEIRUT     2
AUTOZONE             1
GARAGE ELIAS AUTO    1
RYANAIR              1
SPNNYS EXPRESS       1
COSTCO WHOLESALE     1
QATAR AIRWAYS        1
DUBAI MN             1
OOREDOOQATAR.COM     1
CLMNC MEDICAL        1
MALIK BOOKSHOP       1
ALIEXPRESS           1
Name: count, dtype: int64


In [88]:
df['MERCHANT_CITY'] = df['MERCHANT_CITY'].fillna('UNKNOWN')

In [89]:
df['PROCESSING_CODE'].value_counts()


,count
PROCESSING_CODE,
0,2112
20,113
1,71


In [90]:
df['PROCESSING_TYPE'].value_counts()

,count
PROCESSING_TYPE,
Purchase,2112
Purchase Return/Refund,113
ATM Cash Withdrawal,71


In [91]:
pd.crosstab(df['PROCESSING_CODE'], df['PROCESSING_TYPE'])

PROCESSING_TYPE,ATM Cash Withdrawal,Purchase,Purchase Return/Refund
PROCESSING_CODE,,,
0,0,2112,0
1,71,0,0
20,0,0,113


In [92]:
df=df.drop('PROCESSING_TYPE',axis=1)

In [93]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,...,MATCHES_STATUS,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,MERCHANT_NAME,PAYMENT_VIA,COUNTRY_FIXED
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,5411,LB,ZAHLE,...,Match,2022-03-10 00:51:36,2022-03-13,3,False,False,False,BOU KHALIL MKT,DIRECT,False
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,5999,GB,LONDON,...,Match,2022-07-09 17:49:00,2022-07-10,1,False,False,False,BP FUEL,IZ,False
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,7538,US,ONLINE,...,Match,2022-09-14 00:09:34,2022-09-16,2,False,False,False,HRTZ RENTAL,DIRECT,False
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,7538,US,ONLINE,...,Match,2022-04-22 22:52:00,2022-04-23,1,False,False,False,HRTZ RENTAL,DIRECT,False
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,4111,AE,DUBAI,...,Match,2022-06-24 21:19:59,2022-06-25,1,False,False,False,DUBAI MN,DIRECT,False


In [94]:
df['TERMINAL_ID'].value_counts()

,count
TERMINAL_ID,
00000000,1051
597K7MBR,1
DQEP55CS,1
W9MITW88,1
84TX8HRL,1
...,...
5TLX0BRU,1
BJDGTZN1,1
QRRS5W9K,1


In [95]:
zero = df['TERMINAL_ID'] == '00000000'
print(pd.crosstab(zero, df['MERCHANT_CITY'] == 'ONLINE'))

MERCHANT_CITY  False  True 
TERMINAL_ID                
False           1245      0
True             415    636


In [96]:
df['TERMINAL_ID'] = df['TERMINAL_ID'].replace('00000000', np.nan)
df['HAS_TERMINAL'] = df['TERMINAL_ID'].notna()
df = df.drop(columns=['TERMINAL_ID'])

In [97]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,...,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,MERCHANT_NAME,PAYMENT_VIA,COUNTRY_FIXED,HAS_TERMINAL
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,5411,LB,ZAHLE,...,2022-03-10 00:51:36,2022-03-13,3,False,False,False,BOU KHALIL MKT,DIRECT,False,True
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,5999,GB,LONDON,...,2022-07-09 17:49:00,2022-07-10,1,False,False,False,BP FUEL,IZ,False,True
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,7538,US,ONLINE,...,2022-09-14 00:09:34,2022-09-16,2,False,False,False,HRTZ RENTAL,DIRECT,False,False
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,7538,US,ONLINE,...,2022-04-22 22:52:00,2022-04-23,1,False,False,False,HRTZ RENTAL,DIRECT,False,False
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,4111,AE,DUBAI,...,2022-06-24 21:19:59,2022-06-25,1,False,False,False,DUBAI MN,DIRECT,False,True


In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2296 entries, 0 to 2295
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ACCOUNT_ID        2296 non-null   int64         
 1   TXN_ID            2296 non-null   int64         
 2   TXN_TYPE          2296 non-null   object        
 3   TXN_AMOUNT        2296 non-null   float64       
 4   TXN_CCY           2296 non-null   object        
 5   BILLING_AMOUNT    2296 non-null   float64       
 6   FX_RATE           2296 non-null   float64       
 7   MCC_CODE          2296 non-null   int64         
 8   MERCHANT_COUNTRY  2296 non-null   object        
 9   MERCHANT_CITY     2296 non-null   object        
 10  PROCESSING_CODE   2296 non-null   int64         
 11  AUTH_CODE         2296 non-null   object        
 12  MATCHES_STATUS    2296 non-null   object        
 13  TXN_DATE_TIME     2296 non-null   datetime64[ns]
 14  SETTLE_DATE       2296 n

In [99]:
print(df['AUTH_CODE'].nunique(), df['AUTH_CODE'].value_counts().head(10))

2191 AUTH_CODE
000000    71
ZZ0011    16
9X4K7P    12
A1B2C3    10
LDEEDY     1
QKM2RX     1
FEPK2U     1
U6XHPD     1
6J4HFL     1
QNVB9T     1
Name: count, dtype: int64


In [100]:
# 1. are the 000000 all ATM?
print(pd.crosstab(df['AUTH_CODE'] == '000000', df['PROCESSING_CODE']))

# 2. the repeated codes: same transaction copied, or just a fake code?
rep = df['AUTH_CODE'].isin(['ZZ0011', '9X4K7P', 'A1B2C3'])
print(df.loc[rep, ['AUTH_CODE', 'ACCOUNT_ID', 'TXN_AMOUNT', 'MERCHANT_NAME', 'TXN_DATE_TIME']].sort_values('AUTH_CODE').head(20))

PROCESSING_CODE    0   1    20
AUTH_CODE                     
False            2049  67  109
True               63   4    4
     AUTH_CODE  ACCOUNT_ID  TXN_AMOUNT        MERCHANT_NAME  \
43      9X4K7P   587985796     -229.59              REVOLUT   
2101    9X4K7P   926579545      102.56    GARAGE ELIAS AUTO   
1611    9X4K7P   531531981     -336.59        THE BODY SHOP   
1601    9X4K7P   914142388    -1053.43      SAUDIA AIRLINES   
1199    9X4K7P   534461724     -198.40    QUEST DIAGNOSTICS   
809     9X4K7P   846951652     -231.84        TESCO EXPRESS   
706     9X4K7P   720785390      -11.99            GRAMMARLY   
2197    9X4K7P   712485499     -137.52        CHATEAU KSARA   
60      9X4K7P   841486856     -120.47  CARREFOUR CITY MALL   
289     9X4K7P   750360244     -451.46           HOME DEPOT   
275     9X4K7P   872147333     -242.87  SEPHORA MIDDLE EAST   
283     9X4K7P   993428902      -92.52               TOTERS   
1058    A1B2C3   921269072     -752.59    PANDA SUPERMARK

In [101]:
df = df.drop(columns=['AUTH_CODE'])

In [102]:
df.head()

,ACCOUNT_ID,TXN_ID,TXN_TYPE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,FX_RATE,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,...,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,MERCHANT_NAME,PAYMENT_VIA,COUNTRY_FIXED,HAS_TERMINAL
0,716075454,4700058,Settled,-104.39,USD,-104.39,1.000000,5411,LB,ZAHLE,...,2022-03-10 00:51:36,2022-03-13,3,False,False,False,BOU KHALIL MKT,DIRECT,False,True
1,551324309,4700106,Settled,-162.99,USD,-162.99,1.000000,5999,GB,LONDON,...,2022-07-09 17:49:00,2022-07-10,1,False,False,False,BP FUEL,IZ,False,True
2,749053287,4700140,Settled,-480.79,USD,-480.79,1.000000,7538,US,ONLINE,...,2022-09-14 00:09:34,2022-09-16,2,False,False,False,HRTZ RENTAL,DIRECT,False,False
3,671284639,4700183,Settled,-489.74,USD,-489.74,1.000000,7538,US,ONLINE,...,2022-04-22 22:52:00,2022-04-23,1,False,False,False,HRTZ RENTAL,DIRECT,False,False
4,649925833,4700220,Settled,-58.09,AED,-15.54,0.267538,4111,AE,DUBAI,...,2022-06-24 21:19:59,2022-06-25,1,False,False,False,DUBAI MN,DIRECT,False,True


In [103]:
df['MATCHES_STATUS'].value_counts()

,count
MATCHES_STATUS,
Match,2062
Unmatched,140
Pending,94


In [104]:
print('Repeated TXN_ID:', df['TXN_ID'].duplicated().sum())
print('Fully identical rows:', df.duplicated().sum())

Repeated TXN_ID: 0
Fully identical rows: 0


In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2296 entries, 0 to 2295
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ACCOUNT_ID        2296 non-null   int64         
 1   TXN_ID            2296 non-null   int64         
 2   TXN_TYPE          2296 non-null   object        
 3   TXN_AMOUNT        2296 non-null   float64       
 4   TXN_CCY           2296 non-null   object        
 5   BILLING_AMOUNT    2296 non-null   float64       
 6   FX_RATE           2296 non-null   float64       
 7   MCC_CODE          2296 non-null   int64         
 8   MERCHANT_COUNTRY  2296 non-null   object        
 9   MERCHANT_CITY     2296 non-null   object        
 10  PROCESSING_CODE   2296 non-null   int64         
 11  MATCHES_STATUS    2296 non-null   object        
 12  TXN_DATE_TIME     2296 non-null   datetime64[ns]
 13  SETTLE_DATE       2296 non-null   datetime64[ns]
 14  SETTLE_DAYS       2296 n

In [106]:
df.describe()

,ACCOUNT_ID,TXN_ID,TXN_AMOUNT,BILLING_AMOUNT,FX_RATE,MCC_CODE,PROCESSING_CODE,TXN_DATE_TIME,SETTLE_DATE,SETTLE_DAYS
count,2.296000e+03,2.296000e+03,2.296000e+03,2296.000000,2296.000000,2296.000000,2296.000000,2296,2296,2296.000000
mean,7.355624e+08,4.738719e+06,-1.403289e+06,-309.267199,0.744285,5924.650261,1.015244,2022-06-16 20:05:12.510888448,2022-06-18 05:13:35.331010304,1.871516
min,5.018934e+08,4.700058e+06,-1.426670e+08,-6461.790000,0.000026,4111.000000,0.000000,2022-03-01 00:19:00,2022-03-02 00:00:00,0.000000
25%,6.237910e+08,4.719314e+06,-1.565217e+03,-363.885000,0.266717,5411.000000,0.000000,2022-04-22 10:45:40.750000128,2022-04-24 00:00:00,1.000000
50%,7.302085e+08,4.738536e+06,-2.661150e+02,-174.450000,1.000000,5815.000000,0.000000,2022-06-18 05:12:00,2022-06-19 00:00:00,2.000000
75%,8.407364e+08,4.758133e+06,-5.000000e+01,-35.110000,1.000000,6011.000000,0.000000,2022-08-10 06:00:28.500000,2022-08-12 00:00:00,2.000000
max,9.982516e+08,4.777197e+06,1.773980e+07,3774.250000,3.305427,8220.000000,20.000000,2022-09-30 18:52:02,2022-10-03 00:00:00,4.000000
std,1.325324e+08,2.231716e+04,6.747856e+06,613.477312,0.502324,1030.660827,4.323743,NaN,NaN,0.968141


In [107]:
df.to_excel('clean_transactions.xlsx', index=False)


In [108]:

m = df.copy()

m['TXN_HOUR']    = m['TXN_DATE_TIME'].dt.hour
m['TXN_WEEKDAY'] = m['TXN_DATE_TIME'].dt.dayofweek
m['TXN_MONTH']   = m['TXN_DATE_TIME'].dt.month
m = m.drop(columns=['TXN_DATE_TIME', 'SETTLE_DATE'])


m = m.drop(columns=['ACCOUNT_ID', 'TXN_ID', 'TXN_AMOUNT'])



In [109]:
m['TXN_TYPE'] = (m['TXN_TYPE'] == 'Refund').astype(int)
bool_cols = m.select_dtypes('bool').columns
m[bool_cols] = m[bool_cols].astype(int)



In [110]:
def keep_top(col, n):
    top = m[col].value_counts().index[:n]
    m[col] = m[col].where(m[col].isin(top), 'OTHER')

keep_top('TXN_CCY', 8)
keep_top('MERCHANT_COUNTRY', 10)
keep_top('MCC_CODE', 15)


for col in ['MERCHANT_NAME', 'MERCHANT_CITY']:
    m[col + '_FREQ'] = m[col].map(m[col].value_counts())
m = m.drop(columns=['MERCHANT_NAME', 'MERCHANT_CITY'])


m = pd.get_dummies(m, columns=['PAYMENT_VIA', 'TXN_CCY', 'MERCHANT_COUNTRY',
                               'MCC_CODE', 'PROCESSING_CODE'], dtype=int)

m['MATCHES_STATUS'] = m['MATCHES_STATUS'].map({'Match': 0, 'Pending': 1, 'Unmatched': 2})

print(m.shape)
print(m.dtypes.value_counts())

(2296, 62)
int64      57
int32       3
float64     2
Name: count, dtype: int64


In [111]:
m.head()

,TXN_TYPE,BILLING_AMOUNT,FX_RATE,MATCHES_STATUS,SETTLE_DAYS,SETTLE_IMPUTED,BILLING_FIXED,AMT_MISMATCH,COUNTRY_FIXED,HAS_TERMINAL,...,MCC_CODE_5999,MCC_CODE_6011,MCC_CODE_7011,MCC_CODE_7372,MCC_CODE_7399,MCC_CODE_8220,MCC_CODE_OTHER,PROCESSING_CODE_0,PROCESSING_CODE_1,PROCESSING_CODE_20
0,0,-104.39,1.000000,0,3,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
1,0,-162.99,1.000000,0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,0
2,0,-480.79,1.000000,0,2,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
3,0,-489.74,1.000000,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
4,0,-15.54,0.267538,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0


In [112]:
print(m.shape)
print(m.select_dtypes('object').columns.tolist())
print(m.isna().sum().sum())

(2296, 62)
[]
0
